# **Society Of Mind**

In [1]:
from autogen_ext.models.ollama import OllamaChatCompletionClient

## TODO: Local Model Calling.

model_client = OllamaChatCompletionClient(model="qwen2.5:14b", parallel_tool_calls=False)


In [2]:
from autogen_core.models import UserMessage
await model_client.create(
    [UserMessage(content="Hi, i'm Al Amin, Nine to meet you.", source="user")],
)

CreateResult(finish_reason='stop', content='Hello Al Amin! Nice to meet you. How can I assist you today? If you have any questions or need help with anything, feel free to let me know.', usage=RequestUsage(prompt_tokens=42, completion_tokens=36), cached=False, logprobs=None, thought=None)

In [3]:
import asyncio
from autogen_agentchat.ui import Console
from autogen_agentchat.agents import AssistantAgent, SocietyOfMindAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination


async def main() -> None:
    model_client = OllamaChatCompletionClient(model="qwen2.5:14b")

    agent1 = AssistantAgent("assistant1", model_client=model_client, system_message="You are a writer, write well.")
    agent2 = AssistantAgent(
        "assistant2",
        model_client=model_client,
        system_message="You are an editor, provide critical feedback. Respond with 'APPROVE' if the text addresses all feedbacks.",
    )
    inner_termination = TextMentionTermination("APPROVE")
    inner_team = RoundRobinGroupChat([agent1, agent2], termination_condition=inner_termination, max_turns=2)

    society_of_mind_agent = SocietyOfMindAgent("society_of_mind", team=inner_team, model_client=model_client, response_prompt='Output a standalone response to the original request under 50 word, without mentioning any of the intermediate discussion.')

    agent3 = AssistantAgent(
        "assistant3", model_client=model_client, system_message="Translate the text to Spanish."
    )
    team = RoundRobinGroupChat([society_of_mind_agent, agent3], max_turns=2)

    stream = team.run_stream(task="Write a short story with a surprising ending in 100 word.")
    await Console(stream)


await (main())


---------- TextMessage (user) ----------
Write a short story with a surprising ending in 100 word.
---------- TextMessage (assistant1) ----------
Alice found an old map in her grandfather's attic and decided to follow its cryptic clues to a nearby park. As she reached the final spot marked by an 'X', all she found was a small, rusted key. Confused, she pocketed it and walked home. That night, as Alice prepared for bed, she noticed how the key fit perfectly into her grandfather's old locked drawer. Inside, to her astonishment, were childhood treasures belonging not just to him but also to her late grandmother, whom he had never spoken of before.
---------- TextMessage (assistant2) ----------
This short story effectively incorporates a surprising twist at its conclusion that ties back to the setup with the map and key from the grandfather's attic. It successfully surprises by revealing hidden family history through a tangible object (the key). However, it could benefit from more details 